
# Prompts:
First Prompt: need to do data preproccesing on yelp photos dataset

Last Prompt: so here the main folders like food, drink, menu, outside, inside are empty so should i just delete it ?

# Install Dependencies

In [1]:
!pip install pillow opencv-python torchvision torch scikit-learn pandas numpy tqdm --upgrade

  Using cached scikit_learn-1.7.0-cp310-cp310-win_amd64.whl.metadata (14 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.2/39.0 MB 6.7 MB/s eta 0:00:06
    --------------------------------------- 0.6/39.0 MB 5.7 MB/s eta 0:00:07
    --------------------------------------- 0.9/39.0 MB 5.5 MB/s eta 0:00:07
   - -------------------------------------- 1.2/39.0 MB 5.9 MB/s eta 0:00:07
   - -------------------------------------- 1.5/39.0 MB 5.4 MB/s eta 0:00:07
   - -------------------------------------- 1.9/39.0 MB 5.6 MB/s eta 0:00:07
   -- ------------------------------------- 2.2/39.0 MB 5.7 MB/s eta 0:00:07
   -- ------------------------------------- 2.4/39.0 MB 5.4 MB/s eta 0:00:07
   -- ------------------------------------- 2.6/39.0 MB 5.3 MB/s eta 0:00:07
   -- ------------------------------------- 2.8/39.0 MB 5.2 MB/s eta 0:00:07
   --- ------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires keras>=3.5.0, but you have keras 2.11.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 3.19.6 which is incompatible.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.11.2 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 1: Import Libraries and Configure Paths

In [9]:
import json
import os
import random
from collections import defaultdict, Counter
import cv2
import numpy as np
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
import torchvision.transforms as transforms
import tarfile

In [15]:
# Configuration
SOURCE_DIR = r"C:\Users\Meet\Desktop\Yelp Photos"  # Location of the dataset archive
TAR_FILE = os.path.join(SOURCE_DIR, "yelp_photos.tar")
EXTRACT_DIR = r"C:\Users\Meet\Desktop\Yelp2"       # Extraction and processing location
PHOTO_DIR = os.path.join(EXTRACT_DIR, "photos")   # Directory with .jpg files
METADATA_PATH = os.path.join(EXTRACT_DIR, "photos.json")  # Metadata file
OUTPUT_DIR = os.path.join(EXTRACT_DIR, "final_processed_data")  # Output directory
TARGET_SIZE = (224, 224)  # Standard size for CNNs
SAMPLE_FRACTION = 0.1    # 10% stratified sampling
BATCH_SIZE = 1000        # Process images in batches

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [20]:
# Extract tar file if not already extracted
if not os.path.exists(EXTRACT_DIR):
    print(f"Extracting {TAR_FILE} to {EXTRACT_DIR}...")
    with tarfile.open(TAR_FILE, "r") as tar:
        tar.extractall(path=EXTRACT_DIR)
    print("Extraction complete!")
else:
    print(f"{EXTRACT_DIR} already exists, skipping extraction.")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

Extracting C:\Users\Meet\Desktop\Yelp Photos\yelp_photos.tar to C:\Users\Meet\Desktop\Yelp2...
Extraction complete!


# Step 2: Load and Filter Metadata

In [21]:
def load_and_filter_metadata():
    """Load photos.json and filter valid images."""
    try:
        with open(METADATA_PATH, 'r') as f:
            data = [json.loads(line) for line in f]
        print(f"Total photos in metadata: {len(data)}")
        
        valid_data = []
        for item in tqdm(data, desc="Filtering metadata"):
            photo_path = os.path.join(PHOTO_DIR, f"{item['photo_id']}.jpg")
            if os.path.exists(photo_path):
                try:
                    img = Image.open(photo_path)
                    img.verify()  # Check for corruption
                    valid_data.append(item)
                except Exception as e:
                    print(f"Corrupted image: {item['photo_id']}, error: {e}")
            else:
                print(f"Missing image: {item['photo_id']}")
        
        print(f"Valid photos after filtering: {len(valid_data)}")
        
        # Save filtered metadata
        with open(os.path.join(OUTPUT_DIR, "filtered_photos.json"), 'w') as f:
            json.dump(valid_data, f, indent=2)
        
        return valid_data
    except Exception as e:
        print(f"Error loading metadata: {e}")
        return []

# Run step
photos_metadata = load_and_filter_metadata()

Total photos in metadata: 200100


Filtering metadata:   1%|          | 1606/200100 [00:26<1:16:46, 43.09it/s]

Corrupted image: ydm3g1wUWSxJnMPgHk2JhQ, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp2\\photos\\ydm3g1wUWSxJnMPgHk2JhQ.jpg'


Filtering metadata:   2%|▏         | 4234/200100 [01:06<47:57, 68.07it/s]  

Corrupted image: JGpfPj8VEvnq1B-Xqr3w-A, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp2\\photos\\JGpfPj8VEvnq1B-Xqr3w-A.jpg'


...

Corrupted image: iX-8Xm2G7meRHUg8qhoL1A, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp2\\photos\\iX-8Xm2G7meRHUg8qhoL1A.jpg'


Filtering metadata: 100%|█████████▉| 199235/200100 [48:51<00:28, 30.21it/s]

Corrupted image: GWLmPwKeBnh2b_7Kv_LQ7w, error: cannot identify image file 'C:\\Users\\Meet\\Desktop\\Yelp2\\photos\\GWLmPwKeBnh2b_7Kv_LQ7w.jpg'


Filtering metadata: 100%|██████████| 200100/200100 [49:03<00:00, 67.98it/s]


Valid photos after filtering: 199994


# Step 3: Remove menu Class and Perform Stratified Sampling

In [22]:
def remove_menu_and_stratified_sampling(photos_metadata, sample_fraction=0.1):
    """Remove 'menu' class and perform stratified sampling on remaining classes."""
    # Filter out 'menu' class
    filtered_metadata = [item for item in photos_metadata if item['label'] != 'menu']
    print(f"Removed 'menu' class. Remaining photos: {len(filtered_metadata)}")
    
    # Group by label for stratified sampling
    label_to_items = defaultdict(list)
    for item in filtered_metadata:
        label_to_items[item['label']].append(item)
    
    original_counts = {label: len(items) for label, items in label_to_items.items()}
    print("Original label distribution (after removing 'menu'):", original_counts)
    
    sampled_data = []
    for label, items in label_to_items.items():
        k = max(1, int(len(items) * sample_fraction))
        sampled = random.sample(items, k=k)
        sampled_data.extend(sampled)
        print(f"{label}: sampled {k} out of {len(items)}")
    
    random.shuffle(sampled_data)
    print(f"Total images after stratified sampling: {len(sampled_data)}")
    
    with open(os.path.join(OUTPUT_DIR, "sampled_photos.json"), 'w', encoding='utf-8') as f:
        json.dump(sampled_data, f, indent=2)
    
    return sampled_data

# Run step
sampled_metadata = remove_menu_and_stratified_sampling(photos_metadata, SAMPLE_FRACTION)

Removed 'menu' class. Remaining photos: 198316
Original label distribution (after removing 'menu'): {'inside': 56030, 'outside': 18569, 'drink': 15670, 'food': 108047}
inside: sampled 5603 out of 56030
outside: sampled 1856 out of 18569
drink: sampled 1567 out of 15670
food: sampled 10804 out of 108047
Total images after stratified sampling: 19830


# Step 4: Split Data

In [23]:
def initial_split(sampled_metadata):
    """Perform initial split of sampled data into train, val, and test sets."""
    # Group by label for stratified splitting
    label_to_items = defaultdict(list)
    for item in sampled_metadata:
        label_to_items[item['label']].append(item)
    
    train_data = []
    val_data = []
    test_data = []
    
    for label, items in label_to_items.items():
        # Stratified split: 80% train, 10% val, 10% test
        n = len(items)
        n_train = int(0.8 * n)
        n_val = int(0.1 * n)
        n_test = n - n_train - n_val
        
        # Ensure at least 1 sample per split if possible
        n_train = max(1, n_train)
        n_val = max(1, n_val) if n - n_train >= 1 else 0
        n_test = max(1, n_test) if n - n_train - n_val >= 1 else 0
        
        if n_val + n_test > n - n_train:
            n_val = max(0, n - n_train - n_test)
        
        # Perform split
        random.shuffle(items)
        train_data.extend(items[:n_train])
        val_data.extend(items[n_train:n_train + n_val])
        test_data.extend(items[n_train + n_val:])
    
    print(f"Train samples: {len(train_data)}")
    print(f"Validation samples: {len(val_data)}")
    print(f"Test samples: {len(test_data)}")
    
    return train_data, val_data, test_data

# Run step
train_data, val_data, test_data = initial_split(sampled_metadata)

Train samples: 15862
Validation samples: 1981
Test samples: 1987


# Step 5: Balance Classes with Augmentation

In [24]:
def balance_classes(split_data, split_name):
    """Balance classes within a split using augmentation."""
    label_counts = Counter(item['label'] for item in split_data)
    print(f"{split_name} label distribution before balancing:", dict(label_counts))
    
    if not label_counts:
        return split_data, os.path.join(OUTPUT_DIR, split_name)
    
    target_count = int(max(label_counts.values()) * 0.25)
    print(f"{split_name} target count per class: {target_count}")
    
    augment_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(20),
        transforms.RandomResizedCrop(TARGET_SIZE, scale=(0.8, 1.0)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
    ])
    
    balanced_dir = os.path.join(OUTPUT_DIR, split_name)
    os.makedirs(balanced_dir, exist_ok=True)
    
    balanced_data = []
    classes_to_augment = {'outside', 'drink'}  # 'menu' already removed
    
    for label in label_counts:
        label_dir = os.path.join(balanced_dir, label)
        os.makedirs(label_dir, exist_ok=True)
        
        label_items = [item for item in split_data if item['label'] == label]
        current_count = len(label_items)
        
        for item in label_items:
            src = os.path.join(PHOTO_DIR, f"{item['photo_id']}.jpg")
            dst = os.path.join(label_dir, f"{item['photo_id']}.jpg")
            if os.path.exists(src):
                shutil.copyfile(src, dst)
                balanced_data.append(item)
            else:
                print(f"Missing image: {src}")
        
        if label in classes_to_augment and current_count < target_count:
            print(f"Balancing {label} in {split_name}: Current={current_count}, Target={target_count}")
            copies_needed = target_count - current_count
            augment_per_image = max(1, copies_needed // current_count)
            extra_needed = copies_needed - (augment_per_image * current_count)
            
            for idx, item in enumerate(tqdm(label_items, desc=f"Augmenting {label} in {split_name}")):
                src = os.path.join(PHOTO_DIR, f"{item['photo_id']}.jpg")
                if not os.path.exists(src):
                    continue
                img = Image.open(src).convert('RGB')
                
                for i in range(augment_per_image):
                    aug_img = augment_transform(img)
                    aug_id = f"{item['photo_id']}_aug{i}"
                    aug_path = os.path.join(label_dir, f"{aug_id}.jpg")
                    aug_img.save(aug_path, quality=95)
                    balanced_data.append({
                        'photo_id': aug_id,
                        'label': label,
                        'business_id': item['business_id'],
                        'caption': item['caption']
                    })
                
                if idx < extra_needed:
                    aug_img = augment_transform(img)
                    aug_id = f"{item['photo_id']}_extra_aug"
                    aug_path = os.path.join(label_dir, f"{aug_id}.jpg")
                    aug_img.save(aug_path, quality=95)
                    balanced_data.append({
                        'photo_id': aug_id,
                        'label': label,
                        'business_id': item['business_id'],
                        'caption': item['caption']
                    })
    
    final_counts = Counter(item['label'] for item in balanced_data)
    print(f"{split_name} final balanced distribution:", dict(final_counts))
    
    return balanced_data, balanced_dir

# Run step for each split
train_balanced, train_balanced_dir = balance_classes(train_data, 'train')
val_balanced, val_balanced_dir = balance_classes(val_data, 'val')
test_balanced, test_balanced_dir = balance_classes(test_data, 'test')

train label distribution before balancing: {'drink': 1253, 'food': 8643, 'inside': 4482, 'outside': 1484}
train target count per class: 2160
Balancing drink in train: Current=1253, Target=2160


Augmenting drink in train: 100%|██████████| 1253/1253 [00:38<00:00, 32.69it/s]


Balancing outside in train: Current=1484, Target=2160


Augmenting outside in train: 100%|██████████| 1484/1484 [00:30<00:00, 49.29it/s]


train final balanced distribution: {'drink': 2506, 'food': 8643, 'inside': 4482, 'outside': 2968}
val label distribution before balancing: {'drink': 156, 'food': 1080, 'inside': 560, 'outside': 185}
val target count per class: 270
Balancing drink in val: Current=156, Target=270


...

val final balanced distribution: {'drink': 312, 'food': 1080, 'inside': 560, 'outside': 370}
test label distribution before balancing: {'drink': 158, 'food': 1081, 'inside': 561, 'outside': 187}
test target count per class: 270
Balancing drink in test: Current=158, Target=270


Augmenting drink in test: 100%|██████████| 158/158 [00:03<00:00, 50.98it/s]


Balancing outside in test: Current=187, Target=270


Augmenting outside in test: 100%|██████████| 187/187 [00:03<00:00, 53.06it/s]

test final balanced distribution: {'drink': 316, 'food': 1081, 'inside': 561, 'outside': 374}


# Step 6: Preprocess Images

In [25]:
def preprocess_images(balanced_metadata, balanced_dir, split_name):
    """Preprocess images for a given split and save to final structure."""
    processed_images = {}
    failed_images = []
    
    def preprocess_image(image_path):
        try:
            img = cv2.imread(image_path)
            if img is None:
                img = np.array(Image.open(image_path).convert('RGB'))
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            h, w = img.shape[:2]
            scale = min(TARGET_SIZE[0]/h, TARGET_SIZE[1]/w)
            new_h, new_w = int(h * scale), int(w * scale)
            img = cv2.resize(img, (new_w, new_h))
            if new_h != TARGET_SIZE[0] or new_w != TARGET_SIZE[1]:
                delta_h = TARGET_SIZE[0] - new_h
                delta_w = TARGET_SIZE[1] - new_w
                top, bottom = delta_h//2, delta_h-(delta_h//2)
                left, right = delta_w//2, delta_w-(delta_w//2)
                img = cv2.copyMakeBorder(img, top, bottom, left, right, 
                                       cv2.BORDER_CONSTANT, value=[0, 0, 0])
            return img.astype(np.float32) / 255.0
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return None
    
    print(f"Processing images from {balanced_dir} for {split_name}...")
    for item in tqdm(balanced_metadata, desc=f"Processing {split_name} images"):
        photo_id = item['photo_id']
        label = item['label']
        image_path = os.path.join(balanced_dir, label, f"{photo_id}.jpg")
        
        if not os.path.exists(image_path):
            failed_images.append(photo_id)
            continue
        
        processed_img = preprocess_image(image_path)
        if processed_img is not None:
            label_dir = os.path.join(OUTPUT_DIR, split_name, label)
            os.makedirs(label_dir, exist_ok=True)
            output_path = os.path.join(label_dir, f"{photo_id}.jpg")
            cv2.imwrite(
                output_path,
                cv2.cvtColor((processed_img * 255).astype(np.uint8), cv2.COLOR_RGB2BGR),
                [int(cv2.IMWRITE_JPEG_QUALITY), 95]
            )
            processed_images[photo_id] = {
                'path': output_path,
                'label': label,
                'business_id': item['business_id'],
                'caption': item['caption']
            }
    
    with open(os.path.join(OUTPUT_DIR, f"{split_name}_processing_log.json"), 'w', encoding='utf-8') as f:
        json.dump({
            'processed': len(processed_images),
            'failed': len(failed_images),
            'failed_samples': failed_images,
            'class_distribution': {
                label: len([v for v in processed_images.values() if v['label'] == label])
                for label in set(item['label'] for item in balanced_metadata)
            }
        }, f, indent=2)
    
    print(f"{split_name} successfully processed: {len(processed_images)} images")
    print(f"{split_name} failed to process: {len(failed_images)} images")
    if failed_images:
        print(f"First 10 failed images in {split_name}:", failed_images[:10])
    
    return processed_images

# Run step for each split
train_processed = preprocess_images(train_balanced, train_balanced_dir, 'train')
val_processed = preprocess_images(val_balanced, val_balanced_dir, 'val')
test_processed = preprocess_images(test_balanced, test_balanced_dir, 'test')

Processing images from C:\Users\Meet\Desktop\Yelp2\final_processed_data\train for train...


Processing train images: 100%|██████████| 18599/18599 [08:30<00:00, 36.41it/s]


train successfully processed: 18599 images
train failed to process: 0 images
Processing images from C:\Users\Meet\Desktop\Yelp2\final_processed_data\val for val...


Processing val images: 100%|██████████| 2322/2322 [00:57<00:00, 40.70it/s]


val successfully processed: 2322 images
val failed to process: 0 images
Processing images from C:\Users\Meet\Desktop\Yelp2\final_processed_data\test for test...


Processing test images: 100%|██████████| 2332/2332 [00:53<00:00, 43.73it/s]


test successfully processed: 2332 images
test failed to process: 0 images


# Step 7: Prevent Data Leakage

In [28]:
def prevent_data_leakage(train_processed, val_processed, test_processed):
    """Prevent data leakage by ensuring no business_id overlap across splits."""
    all_processed = {'train': train_processed, 'val': val_processed, 'test': test_processed}
    business_to_split = defaultdict(list)
    
    # Assign business_ids to their current splits
    for split_name, processed in all_processed.items():
        for photo_id, info in processed.items():
            business_to_split[info['business_id']].append((photo_id, split_name))
    
    # Check for overlap and reassign if needed
    for business_id, photo_info in business_to_split.items():
        if len(photo_info) > 1:  # Multiple splits contain this business_id
            current_splits = set(split for _, split in photo_info)
            if len(current_splits) > 1:
                print(f"Data leakage detected for business_id {business_id} in splits: {current_splits}")
                # Reassign all photos to the largest split (e.g., train)
                dominant_split = max(current_splits, key=lambda s: sum(1 for _, sp in photo_info if sp == s))
                for photo_id, split in photo_info:
                    if split != dominant_split:
                        all_processed[dominant_split][photo_id] = all_processed[split].pop(photo_id)
                print(f"Reassigned to {dominant_split}")
    
    # Update processed dictionaries
    train_processed = all_processed['train']
    val_processed = all_processed['val']
    test_processed = all_processed['test']
    
    print(f"Train samples after leakage prevention: {len(train_processed)}")
    print(f"Validation samples after leakage prevention: {len(val_processed)}")
    print(f"Test samples after leakage prevention: {len(test_processed)}")
    
    return train_processed, val_processed, test_processed

# Step 8: Create Metadata Files

In [29]:
def create_metadata(train_processed, val_processed, test_processed):
    """Create metadata CSVs for train, val, and test splits."""
    meta_dir = os.path.join(OUTPUT_DIR, "metadata")
    os.makedirs(meta_dir, exist_ok=True)
    
    def generate_metadata(processed_data, split_name):
        metadata = []
        missing_files = 0
        for pid in tqdm(processed_data.keys(), desc=f"Creating {split_name} metadata"):
            if pid not in processed_data:
                missing_files += 1
                continue
            info = processed_data[pid]
            file_path = info['path']
            if not os.path.exists(file_path):
                missing_files += 1
                continue
            rel_path = os.path.relpath(file_path, OUTPUT_DIR).replace("\\", "/")
            metadata.append({
                'photo_id': pid,
                'path': rel_path,
                'absolute_path': os.path.abspath(file_path),
                'label': info['label'],
                'business_id': info['business_id'],
                'caption': info['caption'],
                'split': split_name,
                'file_size': os.path.getsize(file_path),
                'modified_time': os.path.getmtime(file_path)
            })
        if missing_files:
            print(f"Warning: {missing_files} files missing from {split_name} set")
        return pd.DataFrame(metadata)
    
    train_meta = generate_metadata(train_processed, 'train')
    val_meta = generate_metadata(val_processed, 'val')
    test_meta = generate_metadata(test_processed, 'test')
    
    train_meta.to_csv(os.path.join(meta_dir, "train_metadata.csv"), index=False, encoding='utf-8')
    val_meta.to_csv(os.path.join(meta_dir, "val_metadata.csv"), index=False, encoding='utf-8')
    test_meta.to_csv(os.path.join(meta_dir, "test_metadata.csv"), index=False, encoding='utf-8')
    
    combined_meta = pd.concat([train_meta, val_meta, test_meta])
    combined_meta.to_csv(os.path.join(meta_dir, "combined_metadata.csv"), index=False, encoding='utf-8')
    
    stats = {
        'total_samples': len(combined_meta),
        'train_samples': len(train_meta),
        'val_samples': len(val_meta),
        'test_samples': len(test_meta),
        'class_distribution': combined_meta['label'].value_counts().to_dict()
    }
    with open(os.path.join(meta_dir, "dataset_stats.json"), 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    
    print("Metadata creation complete!")
    print(f"Files saved in: {meta_dir}")

# Run step
create_metadata(train_processed, val_processed, test_processed)

Creating test metadata: 100%|██████████| 2332/2332 [00:00<00:00, 2802.41it/s]


Metadata creation complete!
Files saved in: C:\Users\Meet\Desktop\Yelp2\final_processed_data\metadata
